# 04 · XCom & Branching

> **Module 1 · Composer Fundamentals & Model · Lecture 1.7–1.8**

In this notebook you will:

- Push and pull XCom values, and understand why XCom is a **database**, not a dict
- **Measure** the 48 KB XCom limit instead of quoting it
- Learn branch semantics: what gets skipped, what still runs, and why
- Master trigger rules — `all_success`, `none_failed`, `all_done`, `always`
- Build the join-point pattern that every branching DAG needs

## 1. XCom — the channel between tasks

A task's return value is automatically pushed to XCom under the key
`return_value`, and downstream tasks receive it as an argument. You can push
additional keys explicitly.

```python
def count_rows(**context) -> int:
    context["ti"].xcom_push(key="threshold", value=1_000_000)
    return 42                      # -> xcom key "return_value"

def choose_strategy(**context):
    ti = context["ti"]
    rows = ti.xcom_pull(task_ids="count_rows")          # 42
    threshold = ti.xcom_pull(task_ids="count_rows", key="threshold")
    return ["full_load"]
```

### The model that matters

> **XCom is a database, not a dict.**

In real Airflow, XCom values live in the **metadata database** (or an
external XCom backend). That is not an implementation detail — it is the
constraint that shapes every correct XCom usage:

| | a dict | a database |
|---|---|---|
| size | unbounded | row limits, storage cost |
| concurrent access | trivial | transactions, contention |
| reads | free | a query per pull |
| the whole task graph | in RAM | on disk |

Every "why is my DAG slow" question about XCom is answered by "because it is
a database and you are using it like a dict".

## 2. The mini-runner's XCom

The runner implements the same semantics with an in-memory dict — but the
*interface* is identical, so the code you write here is the code you deploy.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

import inspect

print("XCom interface in the mini-runner:\n")
for name in ("push", "pull", "all_for"):
    method = getattr(local_runner.XCom, name)
    params = ", ".join(p for p in inspect.signature(method).parameters if p != "self")
    print(f"  {name}({params})")

print()
print("DagRun.pull:")
print(f"  {inspect.signature(local_runner.DagRun.pull)}")
print()
print("Note the default: key='return_value', default=None.")
print("That default is the single most common XCom bug - a pull with the")
print("wrong key returns None, not an error, and the failure surfaces")
print("somewhere else entirely.")

### Pushing and pulling, hands on

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task


def produce(run, context):
    """Push several keys and return a value."""
    run.xcom.push("produce", "threshold", 1_000_000)
    run.xcom.push("produce", "partition", "20260115")
    run.xcom.push("produce", "rows", [{"order_id": "o1"}, {"order_id": "o2"}])
    return 42                      # becomes key "return_value"


def consume(run, context):
    """Pull them back."""
    returned = run.pull("produce")                      # return_value
    threshold = run.pull("produce", "threshold")
    partition = run.pull("produce", "partition")
    rows = run.pull("produce", "rows")
    return {
        "return_value": returned,
        "threshold": threshold,
        "partition": partition,
        "row_count": len(rows),
    }


demo = DAG(
    "xcom_demo",
    tasks=[
        Task("produce", produce, dependencies=[]),
        Task("consume", consume, dependencies=["produce"]),
    ],
)

run = local_runner.run_dag(demo, logical_date="2026-01-15")
print("States:", run.state)
print()
print("Consume result:", run.pull("consume"))
print()
print("Everything 'produce' pushed:")
for key, value in run.xcom.all_for("produce").items():
    print(f"  {key:<14} {value!r}")

### Namespacing and the pull-with-default trap

XCom is namespaced **per task**. `run.pull("produce", "partition")` returns
`None` if `produce` never pushed a `partition` key. It does **not** raise.

That matters because a `None` propagates silently into arithmetic:
`None + 1` fails far from the cause. In production code, pull with an
explicit default and check it.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

# A pull with the wrong key returns None.
wrong = run.pull("produce", "partiton")     # note the typo
print(f"pull with a typo'd key -> {wrong!r}  (no exception raised)")

# The safe form: pull with an explicit default.
safe = run.pull("produce", "partiton", default="MISSING")
print(f"pull with a default     -> {safe!r}")

# The safest form: check it.
value = run.pull("produce", "partition")
if value is None:
    raise ValueError("partition was not pushed by 'produce'")
print(f"checked pull            -> {value!r}")

## 3. The 48 KB limit — measured, not quoted

Airflow's XCom value size limit is commonly quoted as **48 KB**. Rather than
take it on faith, let's measure the actual behaviour and, more usefully,
measure the *encoded* size of the payloads you would realistically push.

The practical consequence is simple: **XCom is for control data — counts,
partition names, decisions, small references. Not for data.**

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import json

# Measure the serialised size of progressively larger XCom payloads.
LIMIT_BYTES = 48 * 1024


def payload_size(value):
    """Return the UTF-8 byte size of a value as Airflow would store it."""
    return len(json.dumps(value, default=str).encode("utf-8"))


print(
    f"Airflow's commonly-quoted XCom limit: {LIMIT_BYTES:,} bytes "
    f"({LIMIT_BYTES // 1024} KB)\n"
)
print(f"{'payload':<34} {'rows':>8} {'bytes':>10}  {'over limit?':>12}")
print("-" * 70)

cases = [
    ("a single count", 1),
    ("a partition name", 1),
    ("a list of 100 order ids", 100),
    ("a list of 1,000 order ids", 1000),
    ("a list of 5,000 order ids", 5000),
    ("a list of 20,000 order ids", 20000),
]

for label, count in cases:
    if "count" in label or "partition" in label:
        value = 1245 if "count" in label else "20260115"
        rows = 1
    else:
        value = [
            {"order_id": f"o{n:08d}", "amount": 123.45} for n in range(count)
        ]
        rows = count
    size = payload_size(value)
    over = "YES" if size > LIMIT_BYTES else "no"
    print(f"{label:<34} {rows:>8} {size:>10,}  {over:>12}")

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

# How much data actually fits in 48 KB?
def rows_that_fit(sample_row, limit=LIMIT_BYTES):
    """Estimate how many rows of this shape fit in the limit."""
    per_row = payload_size(sample_row)
    return limit // per_row


order_row = {
    "order_id": "o00000000",
    "customer_id": "c000914",
    "order_ts": "2026-01-15T00:54:38+00:00",
    "region": "AMER",
    "channel": "web",
    "currency": "GBP",
    "status": "complete",
    "amount": 700.97,
    "product_id": "p0173",
}

print(f"One realistic order row: {payload_size(order_row):,} bytes")
print(f"Rows that fit in 48 KB    : ~{rows_that_fit(order_row):,}")
print()
print("Our sample has 1245 accepted orders:")
print(
    "  as a full XCom payload  : "
    f"{payload_size([dict(order_row, order_id=f'o{n:08d}') for n in range(1245)]):,} bytes"
)
print()
print("So the entire accepted dataset for one day does NOT fit in one XCom.")
print("It has to go through Cloud Storage or BigQuery, with only the")
print("REFERENCE (the path) travelling via XCom.")
print()
print("Real Airflow behaviour: a value over the limit raises")
print("AirflowXComSizeLimitExceeded and the task FAILS - it does not")
print("silently truncate. That is the safe failure mode.")

### The rule, and the pattern that follows

> **Push the reference, not the data.**

```python
# WRONG: 1,245 rows through the metadata database.
def transform(**context):
    return accepted_rows          # ~250 KB of JSON per run, per task

# RIGHT: write the data, push the pointer.
def transform(**context):
    path = "gs://bucket/orders/{{ ds }}/orders.jsonl"
    write_jsonl(accepted_rows, path)
    return {"path": path, "row_count": len(accepted_rows)}
```

The second version is also **faster**: a BigQuery load from GCS is a bulk
operation, while 1245 XCom rows are 1245 inserts plus a query to read them
back.

### The other XCom anti-patterns

| Anti-pattern | Why it hurts |
|---|---|
| Pushing a DataFrame / list of rows | The 48 KB limit, plus serialisation cost |
| Pushing secrets | XCom is stored in the metadata DB and visible in the UI |
| Pushing unbounded output | A task that returns a different shape each run breaks downstream assumptions |
| Using XCom as a queue | It is not a queue; there is no ordering guarantee across tasks |
| Never cleaning up | XCom rows accumulate; set `xcom_cleanup` / use an external backend |

### What to use instead

| Data | Transport |
|---|---|
| A count, a partition name, a decision | **XCom** — perfect |
| A file path / GCS URI | **XCom** — the reference pattern |
| A row-level dataset | **Cloud Storage** (JSONL/Parquet) + a load job |
| A tabular dataset | **BigQuery** — write, then query |
| A large object | **GCS**, with the object name in XCom |

## 4. The metadata-DB model, made explicit

Let's build a tiny model of the metadata database's XCom table and show why
"XCom is a database" has real consequences.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

class MetadataDB:
    """A minimal model of the table XCom values live in."""

    def __init__(self):
        self.rows = []          # (dag_id, task_id, run_id, key, value, size)
        self.read_count = 0

    def insert(self, dag_id, task_id, run_id, key, value):
        size = len(str(value).encode("utf-8"))
        self.rows.append((dag_id, task_id, run_id, key, value, size))
        return size

    def select(self, dag_id, task_id, run_id, key):
        self.read_count += 1
        for row in self.rows:
            if (
                row[0] == dag_id
                and row[1] == task_id
                and row[2] == run_id
                and row[3] == key
            ):
                return row[4]
        return None

    def total_bytes(self):
        return sum(row[5] for row in self.rows)


db = MetadataDB()
run_id = "scheduled__2026-01-15T00:00:00+00:00"

# One control value.
db.insert("etl_to_gcs", "transform", run_id, "return_value", {"accepted_count": 1245})
# One reference.
db.insert(
    "etl_to_gcs", "transform", run_id, "gcs_path",
    "gs://bucket/orders/2026-01-15/orders.jsonl",
)
# One BAD push: the whole dataset.
db.insert(
    "etl_to_gcs", "transform", run_id, "rows",
    [{"order_id": f"o{n}"} for n in range(1245)],
)

print(f"rows in xcom table : {len(db.rows)}")
print(f"total bytes stored : {db.total_bytes():,}")
print()
for row in db.rows:
    print(f"  key={row[3]:<15} size={row[5]:>9,} bytes")
print()
print("The 'rows' key dominates. And it was ONE push.")
print()
print("Each pull is a SELECT - a database round trip:")
before = db.read_count
for _ in range(5):
    db.select("etl_to_gcs", "transform", run_id, "return_value")
print(f"  5 pulls -> {db.read_count - before} reads against the metadata DB")
print()
print("In a DAG with 100 tasks each pulling one XCom value, that is 100")
print("queries against the same database the scheduler uses. On a loaded")
print("Composer environment, that is measurable latency.")

## 5. Branching

`BranchOperator` returns a **list of task_ids to run**. Every *other*
downstream task is marked **skipped**.

```python
def choose_strategy(**context) -> list[str]:
    rows = context["ti"].xcom_pull(task_ids="count_rows") or 0
    return ["full_load"] if rows > 1_000_000 else ["incremental_load"]
```

Two ideas carry most of the weight:

**Branching is about downstream skip, not about skipping yourself.** The
branch task itself succeeds; it *chooses*.

**The skipped branch's siblings get skipped.** A task downstream of the
branch that is not named is skipped — *unless* it declares a trigger rule
that tolerates skips.

### The branch decision is pure logic

The course keeps the decision in `pipeline_logic.should_run_full_load`,
where it can be tested against every boundary — which is exactly where
branch bugs live (off-by-one on `>` versus `>=`).

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import should_run_full_load

print("should_run_full_load(row_count, threshold) -> row_count > threshold\n")
for rows, threshold in (
    (0, 100), (99, 100), (100, 100), (101, 100), (1_000_000, 1_000_000),
):
    print(
        f"  rows={rows:<10} threshold={threshold:<10} -> "
        f"{should_run_full_load(rows, threshold)}"
    )

print()
print("Note the strict inequality: exactly-at-threshold takes the")
print("incremental path. This is a policy decision, and because it is")
print("a pure function, it is testable at the boundary.")

### A branching DAG, end to end

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from pipeline_logic import transform_orders, summarise_by_region, partition_for_date
from local_runner import DAG, Task


def read_orders(run, context):
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def transform(run, context):
    raw = run.pull("read_orders") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("transform", "accepted", accepted)
    return {"accepted_count": len(accepted), "rejected_count": len(rejected)}


def choose_strategy(run, context):
    """The branch task: returns the list of task_ids that should run."""
    from pipeline_logic import should_run_full_load
    stats = run.pull("transform") or {}
    accepted = int(stats.get("accepted_count", 0))
    threshold = int(context["__variables__"].get("full_load_threshold", 1_000_000))
    chosen = (
        "full_load"
        if should_run_full_load(accepted, threshold)
        else "incremental_load"
    )
    return [chosen]


def full_load(run, context):
    accepted = run.pull("transform", "accepted") or []
    return f"full_load: rewrote {len(accepted)} rows"


def incremental_load(run, context):
    accepted = run.pull("transform", "accepted") or []
    partition = partition_for_date(run.logical_date)
    return f"incremental_load: appended {len(accepted)} rows to {partition}"


def finalise(run, context):
    """The join point - depends on BOTH branches."""
    return run.pull("full_load") or run.pull("incremental_load") or "nothing"


branch_dag = DAG(
    "branching_demo",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
        Task(
            "choose_strategy",
            choose_strategy,
            dependencies=["transform"],
            branch=True,
        ),
        Task("full_load", full_load, dependencies=["choose_strategy"]),
        Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
        Task(
            "finalise",
            finalise,
            dependencies=["full_load", "incremental_load"],
            trigger_rule="none_failed",
        ),
    ],
)

run = local_runner.run_dag(
    branch_dag,
    logical_date="2026-01-15",
    variables=local_runner.DEFAULT_VARIABLES,
)
print()
print("States:")
for task_id, state in run.state.items():
    print(f"  {task_id:<20} {state}")
print()
print("finalise ->", run.pull("finalise"))

Read that state table carefully. This is the single most important
table in the notebook:

```text
read_orders       success
transform         success
choose_strategy   success     <- the branch task itself SUCCEEDS
full_load         skipped     <- not chosen
incremental_load  success     <- chosen
finalise          success     <- ran, because its trigger_rule is none_failed
```

The branch task **succeeded**. Branching is not failure and not skipping
yourself. The *other* branch was skipped. And `finalise` ran anyway,
because `none_failed` treats a skipped dependency as acceptable.

### What happens with the default trigger rule?

Let's remove `none_failed` and see the classic bug: a DAG that looks correct
and quietly does nothing.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task

# Rebuild the same DAG but with the default trigger rule on the join.
buggy = DAG(
    "branching_bug",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
        Task(
            "choose_strategy",
            choose_strategy,
            dependencies=["transform"],
            branch=True,
        ),
        Task("full_load", full_load, dependencies=["choose_strategy"]),
        Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
        # NOTE: trigger_rule left at the default "all_success".
        Task(
            "finalise",
            finalise,
            dependencies=["full_load", "incremental_load"],
        ),
    ],
)

run_buggy = local_runner.run_dag(
    buggy,
    logical_date="2026-01-15",
    variables=local_runner.DEFAULT_VARIABLES,
)
print()
print("With the DEFAULT trigger rule:")
for task_id, state in run_buggy.state.items():
    print(f"  {task_id:<20} {state}")
print()
print("finalise ->", run_buggy.pull("finalise"))
print()
print("THE BUG: one branch was skipped, so 'all_success' was not met,")
print("so finalise was skipped too. The pipeline does nothing and the")
print("DAG still reports success. This is the most common branching bug.")

## 6. Trigger rules

| Rule | Runs when | Typical use |
|---|---|---|
| `all_success` (default) | every upstream task succeeded | the normal case |
| `none_failed` | no upstream task **failed** (skips are OK) | **join points** after a branch |
| `none_failed_min_one_success` | no failures and at least one success | conditional cleanup |
| `all_done` | all upstream tasks finished, whatever the outcome | always-run cleanup / alerting |
| `always` | unconditionally | notifications, fan-out |

The distinction that matters:

- **skipped** is not **failed**. `none_failed` tolerates skips; `all_success`
  does not.
- `all_done` and `always` also run after failures, which is what makes them
  right for cleanup and alerting — and wrong for data processing.

Let's compare all four on the same graph.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task


def make_dag(rule):
    return DAG(
        f"rule_{rule}",
        tasks=[
            Task("read_orders", read_orders, dependencies=[]),
            Task("transform", transform, dependencies=["read_orders"]),
            Task(
                "choose_strategy",
                choose_strategy,
                dependencies=["transform"],
                branch=True,
            ),
            Task("full_load", full_load, dependencies=["choose_strategy"]),
            Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
            Task(
                "finalise",
                finalise,
                dependencies=["full_load", "incremental_load"],
                trigger_rule=rule,
            ),
        ],
    )


print(f"{'trigger_rule':<28} {'finalise state':<18} {'finalise output'}")
print("-" * 78)

for rule in ("all_success", "none_failed", "all_done", "always"):
    r = local_runner.run_dag(
        make_dag(rule),
        logical_date="2026-01-15",
        variables=local_runner.DEFAULT_VARIABLES,
    )
    print(
        f"{rule:<28} {r.state.get('finalise', '?'):<18} {r.pull('finalise')!r}"
    )

### When an upstream task *fails*

The rules change again. With a failing upstream:
- `all_success`, `none_failed`, `none_failed_min_one_success` → the task is
  **not run**; Airflow reports `upstream_failed`.
- `all_done`, `always` → the task **does run**.

`upstream_failed` is a **distinct state** from `skipped`, and the distinction
matters when you read a failed DAG: `skipped` means "we chose not to run
this"; `upstream_failed` means "we wanted to run this but could not".

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task


def explode(run, context):
    """A task that always fails."""
    raise RuntimeError("boom - simulated upstream failure")


def always_runs(run, context):
    return "cleanup ran"


print("Upstream 'explode' fails. What happens downstream?\n")
for rule in ("all_success", "none_failed", "all_done", "always"):
    d = DAG(
        f"fail_{rule}",
        tasks=[
            Task("explode", explode, dependencies=[]),
            Task(
                "downstream",
                always_runs,
                dependencies=["explode"],
                trigger_rule=rule,
            ),
        ],
    )
    r = local_runner.run_dag(d, logical_date="2026-01-15")
    print(
        f"  {rule:<28} explode={r.state['explode']:<8} "
        f"downstream={r.state['downstream']}"
    )

### A task with multiple dependencies

`all_success` means **all** must succeed. `none_failed` means **none**
failed (any number may be skipped). `all_done` means they all finished.

A join point that depends on *two independent branches* needs `none_failed`
or `all_done` — never `all_success`.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

print("Join points and their dependency states:\n")
print(
    f"{'dep1':<12} {'dep2':<16} {'all_success':<14} "
    f"{'none_failed':<14} {'all_done'}"
)
print("-" * 70)
for dep1, dep2 in (
    ("success", "success"),
    ("success", "skipped"),
    ("success", "failed"),
    ("skipped", "skipped"),
):
    states = {"a": dep1, "b": dep2}
    all_success = "run" if all(s == "success" for s in states.values()) else "skip"
    none_failed = "run" if "failed" not in states.values() else "skip"
    all_done = "run"
    print(
        f"{dep1:<12} {dep2:<16} {all_success:<14} "
        f"{none_failed:<14} {all_done}"
    )

## 7. The join-point pattern

The rule, stated once:

> **A task that depends on both sides of a branch must declare
> `trigger_rule="none_failed"` (or `"all_done"`).**

Airflow's own docs call this a "join". The course's
`04_branching_and_xcom.py` uses `EmptyOperator` for it:

```python
join = EmptyOperator(
    task_id="load_complete",
    trigger_rule="none_failed",
)
```

`EmptyOperator` is a task with no work — it exists purely to be a
dependency target. That is a legitimate and common pattern.

Let's also cover the **short-circuit** pattern: skip everything downstream
with one task.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task, TaskSkip


def only_if_enough(run, context):
    """A short-circuit: skip everything downstream if the condition fails."""
    accepted = int((run.pull("transform") or {}).get("accepted_count", 0))
    if accepted < 100:
        raise TaskSkip(f"only {accepted} rows, need at least 100")
    return f"{accepted} rows, proceeding"


short_circuit_dag = DAG(
    "short_circuit_demo",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
        Task("check_threshold", only_if_enough, dependencies=["transform"]),
        Task("load", full_load, dependencies=["check_threshold"]),
    ],
)

r = local_runner.run_dag(
    short_circuit_dag,
    logical_date="2026-01-15",
    variables=local_runner.DEFAULT_VARIABLES,
)
print()
print("With 1245 rows (above the threshold):")
for task_id, state in r.state.items():
    print(f"  {task_id:<20} {state}")

## 8. Branching with a forced decision

A real branching DAG often needs an operator to *override* the automatic
decision — "this backfill is a full reload regardless of row count". The
course's `04_branching_and_xcom.py` supports exactly that with a
`force_full_load` param.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

import local_runner
print("local_runner loaded from:", local_runner.__file__)

from local_runner import DAG, Task


def choose_strategy_with_override(run, context):
    """The branch task, with an operator override."""
    from pipeline_logic import should_run_full_load
    stats = run.pull("transform") or {}
    accepted = int(stats.get("accepted_count", 0))
    threshold = int(context["__variables__"].get("full_load_threshold", 1_000_000))
    force = bool(context["params"].get("force_full_load", False))
    if force or should_run_full_load(accepted, threshold):
        return ["full_load"]
    return ["incremental_load"]


override_dag = DAG(
    "branching_override",
    tasks=[
        Task("read_orders", read_orders, dependencies=[]),
        Task("transform", transform, dependencies=["read_orders"]),
        Task(
            "choose_strategy",
            choose_strategy_with_override,
            dependencies=["transform"],
            branch=True,
        ),
        Task("full_load", full_load, dependencies=["choose_strategy"]),
        Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
        Task(
            "finalise",
            finalise,
            dependencies=["full_load", "incremental_load"],
            trigger_rule="none_failed",
        ),
    ],
)

for label, params in (
    ("automatic (1245 < 1M threshold)", {}),
    ("forced full load", {"force_full_load": True}),
):
    r = local_runner.run_dag(
        override_dag,
        logical_date="2026-01-15",
        params=params,
        variables=local_runner.DEFAULT_VARIABLES,
    )
    chosen = [
        t
        for t, s in r.state.items()
        if s == "success" and t in ("full_load", "incremental_load")
    ]
    print(
        f"{label:<36} -> {chosen[0]:<18} finalise={r.state['finalise']}"
    )

## Exercises

1. **XCom budget.** Write a function `xcom_budget(payload)` that returns the JSON byte size of a payload, and use it to compute how many of *your* course's order rows would fit in 48 KB. Then state, in one sentence, what you would push through XCom instead.
2. **Harder — the unskipped branch.** Build a DAG where `full_load` and `incremental_load` both depend on `choose_strategy`, and a `finalise` task depends on both with `trigger_rule="all_done"`. Run it with a low threshold so `full_load` is chosen. Report the state of every task and explain why `finalise` ran even though `incremental_load` was skipped.
3. **Harder — a branch that depends on a branch.** Build a DAG: `choose_a` branches to `path_x` or `path_y`; `choose_b` depends on **both** `path_x` and `path_y` and branches to `final_1` or `final_2`. Run it with the threshold set so `path_x` is chosen. Explain what happens to `choose_b` under each of the trigger rules `all_success`, `none_failed` and `all_done`, and which one you would use.

<details>
<summary><b>Solution (click to reveal)</b></summary>

```python
# Exercise 1
import json


def xcom_budget(payload):
    return len(json.dumps(payload, default=str).encode("utf-8"))


row = {
    "order_id": "o00000000",
    "customer_id": "c000914",
    "order_ts": "2026-01-15T00:54:38+00:00",
    "region": "AMER",
    "channel": "web",
    "currency": "GBP",
    "status": "complete",
    "amount": 700.97,
    "product_id": "p0173",
}

print(xcom_budget([row] * 1245))           # ~300,000+ bytes - way over 48 KB
print(48 * 1024 // xcom_budget([row]))     # ~160 rows fit
```

You would push the **GCS path** of the written file (plus a row count) —
a few dozen bytes — and let the downstream task read the data from GCS or
BigQuery.

```python
# Exercise 2
from local_runner import DAG, Task


def make_join_dag(rule):
    return DAG(
        f"join_{rule}",
        tasks=[
            Task("read_orders", read_orders, dependencies=[]),
            Task("transform", transform, dependencies=["read_orders"]),
            Task(
                "choose_strategy",
                choose_strategy,
                dependencies=["transform"],
                branch=True,
            ),
            Task("full_load", full_load, dependencies=["choose_strategy"]),
            Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
            Task(
                "finalise",
                finalise,
                dependencies=["full_load", "incremental_load"],
                trigger_rule=rule,
            ),
        ],
    )


r = local_runner.run_dag(
    make_join_dag("all_done"),
    logical_date="2026-01-15",
    variables={**local_runner.DEFAULT_VARIABLES, "full_load_threshold": 100},
)
for task_id, state in r.state.items():
    print(f"{task_id:<20} {state}")
```

`full_load` runs, `incremental_load` is skipped, and `finalise` runs
because `all_done` means "run once all upstream tasks have *finished*,
whatever their outcome". `all_success` would have skipped `finalise`
(skipped is not success), and `none_failed` would also have run it (skip is
not failure) — but `all_done` is the stronger statement, because it also
covers the case where one of the branches *failed*.

```python
# Exercise 3
def make_nested(rule):
    return DAG(
        f"nested_{rule}",
        tasks=[
            Task("read_orders", read_orders, dependencies=[]),
            Task("transform", transform, dependencies=["read_orders"]),
            Task(
                "choose_a",
                choose_strategy,
                dependencies=["transform"],
                branch=True,
            ),
            Task("path_x", full_load, dependencies=["choose_a"]),
            Task("path_y", incremental_load, dependencies=["choose_a"]),
            Task(
                "choose_b",
                choose_strategy_with_override,
                dependencies=["path_x", "path_y"],
                branch=True,
                trigger_rule=rule,
            ),
            Task("final_1", full_load, dependencies=["choose_b"]),
            Task("final_2", incremental_load, dependencies=["choose_b"]),
        ],
    )
```

With `path_x` chosen, `path_y` is skipped. Now `choose_b` depends on both:
- `all_success` — `choose_b` is **skipped**, because `path_y` was skipped.
  Nothing downstream runs.
- `none_failed` — `choose_b` **runs** (a skip is not a failure) and can
  branch to `final_1` / `final_2`.
- `all_done` — `choose_b` **runs**, and additionally would run even if
  `path_x` had failed.

For a nested branch I would use `none_failed`: the semantic you want is
"proceed as long as nothing actually broke", and `all_done` would mask a
genuine failure of `path_x` by letting `choose_b` proceed on a skipped and
failed input.

</details>